# Notebook 09 — Parameter Estimation and Model Fitting

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 9 of 12  
**Time estimate:** 90 minutes

> A biological model is only useful if it can be fitted to data. This notebook
> covers two approaches: deterministic least-squares fitting (fast, finds a point
> estimate) and Bayesian MCMC (slower, quantifies uncertainty). Both are applied
> to fitting SIR and Lotka-Volterra parameters to synthetic incidence data.

---
## Step 1 — Motivation

During COVID-19, governments needed estimates of β (transmission rate) and γ
(recovery rate) to project hospital capacity. These were estimated by fitting
SIR (or SEIR) models to reported case counts. But reported cases are incomplete
(only a fraction of infections are tested), noisy (test delays, weekend reporting
artifacts), and β changes over time (interventions). How do you fit a model to
such data, and how do you know whether the fitted parameters are trustworthy?

---
## Step 2 — Intuition

**Least-squares fitting:** choose parameters θ that minimize the sum of squared
residuals between model output y(t; θ) and observed data y_obs(t). For ODE models,
this is done via a numerical optimizer (scipy.optimize.minimize).

**Profile likelihood:** fit the model with one parameter fixed at a range of values,
optimizing all other parameters. The profile likelihood reveals:
- Whether the parameter is identifiable (a clear minimum exists)
- Confidence intervals (95% CI = where profile likelihood drops by 1.92 from the max)

**Bayesian MCMC (Metropolis-Hastings):**
- Define a prior P(θ) — your belief before seeing data
- Define a likelihood P(data | θ) — how probable is the data given parameters?
- MCMC samples from the posterior P(θ | data) ∝ P(data | θ) P(θ)
- Output: a distribution over parameters — uncertainty quantification, not just
  a point estimate
- Metropolis rule: accept proposed θ' with probability min(1, P(θ'|data)/P(θ|data))

---
## Step 3 — Biological Background

**Structural identifiability vs. practical identifiability:**
- *Structural:* can the parameters in principle be uniquely determined from perfect,
  noise-free data? (A property of the model structure, not the data.)
- *Practical:* can they be determined from real, noisy, incomplete data?
  Parameters may be structurally identifiable but practically unidentifiable if
  the data is too noisy or the experiment doesn't observe the right variables.

**SIR identifiability:** β and γ are jointly identifiable if we observe both
I(t) and R(t). If we only observe new cases (= dR/dt approximately), then β/γ
(= R₀) is identifiable but β and γ individually are not from the epidemic curve
alone — we need an independent estimate of γ (e.g., from clinical data on
infection duration).

**Observation models:**
- Poisson observation: reported cases ~ Poisson(I(t) × p_report)
  (appropriate for count data with low mean)
- Gaussian observation: appropriate for continuous measurements, large counts
- Negative binomial: overdispersed count data (COVID-19 case counts, where
  variance >> mean due to superspreading)

---
## Step 4 — Mathematical Explanation

**Likelihood for Poisson observation model:**
If observed counts $y_t \sim \text{Poisson}(\lambda_t(\theta))$:
$$\log P(\mathbf{y} | \theta) = \sum_t \left[ y_t \log \lambda_t(\theta) - \lambda_t(\theta) - \log(y_t!) \right]$$
where $\lambda_t(\theta) = I(t; \theta) \cdot p_{\text{report}} \cdot N$.

**Metropolis-Hastings:**
1. Start from $\theta^{(0)}$
2. Propose $\theta' = \theta^{(t)} + \epsilon$, $\epsilon \sim \mathcal{N}(0, \Sigma)$
3. Acceptance ratio: $\alpha = \min\left(1, \frac{P(\mathbf{y}|\theta') P(\theta')}{P(\mathbf{y}|\theta^{(t)}) P(\theta^{(t)})}\right)$
4. With probability α: $\theta^{(t+1)} = \theta'$; else $\theta^{(t+1)} = \theta^{(t)}$
5. Repeat; discard first burnin samples.

**Key MCMC diagnostics:**
- Acceptance rate: should be 20–50% (tune step size $\Sigma$)
- Trace plot: should look like "white noise" around the posterior mean
- Effective sample size (ESS): accounts for autocorrelation in the chain
- R-hat: split-chain convergence diagnostic (should be < 1.01)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from scipy.stats import poisson as scipy_poisson

rng = np.random.default_rng(42)

# ---- True SIR parameters and synthetic data ----
N_POP = 10000
BETA_TRUE, GAMMA_TRUE = 0.30, 0.10
T_OBS = np.arange(0, 120, 1.0)  # daily observation times
P_REPORT = 0.5  # only 50% of cases reported

def sir_rhs(t, y, beta, gamma, N):
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

def solve_sir(beta, gamma, N=N_POP, I0=1, T_max=120):
    sol = solve_ivp(sir_rhs, (0, T_max), [N-I0, I0, 0],
                    args=(beta, gamma, N),
                    t_eval=np.arange(0, T_max, 1.0),
                    method='RK45', max_step=0.5)
    return sol.t, sol.y  # y: (3, T)

t_true, y_true = solve_sir(BETA_TRUE, GAMMA_TRUE)
# New cases = ΔR per day ≈ γ * I(t)
new_cases_true = np.diff(y_true[2]) * P_REPORT  # incidence
# Observe: Poisson noise around true new cases
y_obs = rng.poisson(np.maximum(new_cases_true, 0.1)).astype(float)
t_obs = t_true[1:]  # one fewer point than t_true

print(f'Peak true new cases: {new_cases_true.max():.0f}/day')
print(f'Peak observed cases: {y_obs.max():.0f}/day')
print(f'Total observed cases: {y_obs.sum():.0f}')

# ---- Least-squares fitting ----
def model_incidence(params, N=N_POP):
    """Solve SIR and return daily incidence (new cases)."""
    beta, gamma = params
    if beta <= 0 or gamma <= 0 or beta > 2 or gamma > 1:
        return None
    try:
        _, y = solve_sir(beta, gamma, N)
        incidence = np.diff(y[2]) * P_REPORT
        return incidence
    except Exception:
        return None

def neg_log_likelihood_poisson(params):
    """Negative log-likelihood under Poisson observation model."""
    inc = model_incidence(params)
    if inc is None or np.any(inc < 0):
        return 1e10
    lam = np.maximum(inc, 1e-9)
    return -np.sum(scipy_poisson.logpmf(y_obs.astype(int), lam))

# Optimize
result = minimize(neg_log_likelihood_poisson,
                    x0=[0.2, 0.15],
                    method='Nelder-Mead',
                    options={'maxiter': 2000, 'xatol': 1e-4})
beta_fit, gamma_fit = result.x
print(f'\nLeast-squares fit: β={beta_fit:.4f} (true={BETA_TRUE}), γ={gamma_fit:.4f} (true={GAMMA_TRUE})')
print(f'Fitted R0={beta_fit/gamma_fit:.2f} (true R0={BETA_TRUE/GAMMA_TRUE:.2f})')

In [ ]:
# ---- Metropolis-Hastings MCMC ----
def log_posterior(params):
    """Log posterior = log likelihood + log prior."""
    beta, gamma = params
    # Log prior: log-uniform on [0.01, 1.0]
    if not (0.01 < beta < 1.5 and 0.01 < gamma < 0.5):
        return -np.inf
    log_prior = 0.0  # uniform prior (improper over valid range)
    log_lik = -neg_log_likelihood_poisson(params)
    return log_prior + log_lik

def metropolis_hastings(log_post_fn, theta0, n_samples, proposal_std, rng):
    """Simple Metropolis-Hastings sampler."""
    theta = np.array(theta0, dtype=float)
    samples = [theta.copy()]
    n_accept = 0
    lp_current = log_post_fn(theta)
    for i in range(n_samples - 1):
        # Proposal
        theta_prop = theta + rng.normal(0, proposal_std, len(theta))
        lp_prop = log_post_fn(theta_prop)
        # Acceptance
        log_ratio = lp_prop - lp_current
        if np.log(rng.random()) < log_ratio:
            theta = theta_prop
            lp_current = lp_prop
            n_accept += 1
        samples.append(theta.copy())
    samples = np.array(samples)
    acc_rate = n_accept / n_samples
    return samples, acc_rate

N_MCMC   = 3000
BURNIN   = 500
PROP_STD = np.array([0.015, 0.008])

print('Running MCMC...')
samples, acc_rate = metropolis_hastings(
    log_posterior, [beta_fit, gamma_fit], N_MCMC, PROP_STD, rng)
print(f'MCMC acceptance rate: {acc_rate:.3f} (target: 0.20-0.45)')

posterior = samples[BURNIN:]
beta_post  = posterior[:, 0]
gamma_post = posterior[:, 1]
R0_post    = beta_post / gamma_post

print(f'Posterior β: {beta_post.mean():.4f} ± {beta_post.std():.4f} (true={BETA_TRUE})')
print(f'Posterior γ: {gamma_post.mean():.4f} ± {gamma_post.std():.4f} (true={GAMMA_TRUE})')
print(f'Posterior R0: {R0_post.mean():.3f} [95% CI: {np.percentile(R0_post,2.5):.3f}–{np.percentile(R0_post,97.5):.3f}]')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel A: Data vs. fitted model
ax = axes[0, 0]
ax.bar(t_obs, y_obs, alpha=0.5, color='grey', width=0.8, label='Observed (noisy)')
ax.plot(t_true[1:], new_cases_true, 'k-', lw=2, label='True incidence')
_, y_fit = solve_sir(beta_fit, gamma_fit)
ax.plot(t_true[1:], np.diff(y_fit[2])*P_REPORT, 'tomato', lw=2, ls='--', label='Least-squares fit')
# 30 posterior predictive samples
for _ in range(30):
    b, g = rng.choice(len(beta_post)), 0
    b = beta_post[rng.integers(len(beta_post))]
    g = gamma_post[rng.integers(len(gamma_post))]
    _, y_s = solve_sir(b, g)
    ax.plot(t_true[1:], np.diff(y_s[2])*P_REPORT, 'steelblue', lw=0.3, alpha=0.2)
ax.set_xlabel('Day'); ax.set_ylabel('New cases/day')
ax.set_title('A. SIR fit to synthetic data\n(observed, true, fitted, posterior predictive)')
ax.legend(fontsize=7)

# Panel B: MCMC trace
ax = axes[0, 1]
ax.plot(samples[:, 0], 'steelblue', lw=0.8, alpha=0.7, label='β')
ax_r = ax.twinx()
ax_r.plot(samples[:, 1], 'tomato', lw=0.8, alpha=0.7, label='γ')
ax.axvline(BURNIN, color='k', ls=':', lw=1, label='Burn-in')
ax.set_xlabel('MCMC step'); ax.set_ylabel('β', color='steelblue')
ax_r.set_ylabel('γ', color='tomato')
ax.set_title('B. MCMC trace plot\n(should mix well after burn-in)')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax_r.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=7)

# Panel C: Posterior joint distribution
ax = axes[0, 2]
ax.scatter(beta_post, gamma_post, s=2, alpha=0.3, color='steelblue')
ax.plot(BETA_TRUE, GAMMA_TRUE, 'r*', ms=14, label='True values')
ax.plot(beta_fit, gamma_fit, 'ko', ms=8, label='LS estimate')
ax.set_xlabel('β'); ax.set_ylabel('γ')
ax.set_title('C. Joint posterior P(β, γ | data)')
ax.legend(fontsize=8)

# Panel D: Marginal posteriors
ax = axes[1, 0]
ax.hist(beta_post, bins=40, density=True, alpha=0.7, color='steelblue', label='P(β|data)')
ax.axvline(BETA_TRUE, color='k', ls='--', lw=1.5, label=f'True β={BETA_TRUE}')
ax.axvline(beta_fit, color='tomato', ls=':', lw=1.5, label=f'LS β={beta_fit:.3f}')
ax.set_xlabel('β'); ax.set_ylabel('Density')
ax.set_title('D. Marginal posterior P(β|data)')
ax.legend(fontsize=8)

ax = axes[1, 1]
ax.hist(R0_post, bins=40, density=True, alpha=0.7, color='green', label='P(R₀|data)')
ax.axvline(BETA_TRUE/GAMMA_TRUE, color='k', ls='--', lw=1.5, label=f'True R₀={BETA_TRUE/GAMMA_TRUE:.1f}')
ax.axvline(beta_fit/gamma_fit, color='tomato', ls=':', lw=1.5, label=f'LS R₀={beta_fit/gamma_fit:.2f}')
q025, q975 = np.percentile(R0_post, [2.5, 97.5])
ax.axvspan(q025, q975, alpha=0.2, color='green', label=f'95% CI [{q025:.2f}, {q975:.2f}]')
ax.set_xlabel('R₀'); ax.set_ylabel('Density')
ax.set_title('E. Posterior distribution of R₀')
ax.legend(fontsize=7)

# Panel F: Profile likelihood for beta
beta_profile_vals = np.linspace(0.15, 0.50, 25)
nll_profile = []
for bv in beta_profile_vals:
    # Optimize over gamma with beta fixed
    res = minimize(lambda g: neg_log_likelihood_poisson([bv, g[0]]),
                    x0=[0.1], method='Nelder-Mead',
                    options={'maxiter': 500})
    nll_profile.append(res.fun)
nll_profile = np.array(nll_profile)
ax = axes[1, 2]
nll_rel = nll_profile - nll_profile.min()
ax.plot(beta_profile_vals, nll_rel, 'steelblue', lw=2)
ax.axhline(1.92, color='tomato', ls='--', lw=1.5, label='95% CI threshold (Δ=1.92)')
ax.axvline(BETA_TRUE, color='k', ls='--', lw=1, label=f'True β={BETA_TRUE}')
critical_beta = beta_profile_vals[nll_rel <= 1.92]
if len(critical_beta) > 0:
    ax.axvspan(critical_beta[0], critical_beta[-1], alpha=0.2, color='steelblue',
                 label=f'95% CI [{critical_beta[0]:.2f}, {critical_beta[-1]:.2f}]')
ax.set_xlabel('β'); ax.set_ylabel('ΔNLL (relative to minimum)')
ax.set_title('F. Profile likelihood for β\n(90% CI = where ΔNLL < 1.92)')
ax.legend(fontsize=7)

plt.suptitle('Module 15 NB09: Parameter Estimation and Model Fitting', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('parameter_estimation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Repeat the fitting with only 30 days of data (early epidemic). Do the
   posterior widths change? Is β more or less identifiable from early data?
2. Fit the Lotka-Volterra model to synthetic prey-predator time series. Report
   the MCMC posterior for all four parameters (r, a, b, d). Which parameters
   are more identifiable?
3. Show the effect of report fraction p on identifiability. Fix β=0.3, γ=0.1
   but fit with p_report=0.5, 0.2, 0.1. Does the posterior for R₀ remain
   correct even when p is misspecified?

---
## Step 10 — Quiz

1. What is the difference between structural and practical identifiability?
   Give an example of a model that is structurally but not practically identifiable.
2. In the Metropolis-Hastings algorithm, when do we reject a proposed sample?
   Why do we sometimes accept a worse (lower likelihood) sample?
3. What does the profile likelihood tell you that a posterior distribution does not?
4. If the MCMC trace plot shows slow drift (the chain doesn't mix — stays in
   one region for hundreds of steps), what is the likely cause and how do you fix it?

---
## Step 12 — Reflection

> *[In one paragraph: explain why fitting β and γ individually from epidemic
> incidence data alone is harder than fitting R₀ = β/γ. What additional data
> would you need to identify β and γ separately? How was this resolved in
> practice during the COVID-19 pandemic?]*

---
*Next: `10_sensitivity_analysis_bifurcation.ipynb`*